In [ ]:
!pip install pandas

In [ ]:
import requests
import pandas as pd
import time
import statsmodels.api as sm

# Preparing crypto coin data

In [ ]:
url = "https://api.binance.com/api/v3/klines"

pairs = [
   "BTCUSDT", "ETHUSDT", "XRPUSDT", "BCHUSDT", "LTCUSDT", "EOSUSDT", "BNBUSDT", "XTZUSDT", "QTUMUSDT", "RVNUSDT", "ZRXUSDT", "ALGOUSDT", "HOTUSDT", "THETAUSDT", "ZENUSDT", "KNCUSDT", "FETUSDT", "CHZUSDT", "FUNUSDT", "LRCUSDT", "HBARUSDT", "ADAUSDT", "DGBUSDT", "SNXUSDT", "ZILUSDT", "DCRUSDT", "ICXUSDT", "LSKUSDT", "ENJUSDT", "PAXGUSDT", "RLCUSDT", "RSRUSDT", "ARDRUSDT", "KAVAUSDT", "SXPUSDT", "TFUELUSDT", "DATAUSDT", "DOTUSDT", "LINKUSDT", "YFIUSDT", "COMPUSDT", "ZECUSDT", "SUSHIUSDT", "KSMUSDT", "UMAUSDT", "EGLDUSDT", "ONTUSDT", "LUNAUSDT", "STXUSDT", "BATUSDT", "AVAXUSDT", "NEARUSDT", "STORJUSDT", "ANKRUSDT", "CVCUSDT", "UTKUSDT", "WANUSDT", "JSTUSDT", "HIVEUSDT", "FTMUSDT", "IOTXUSDT", "DOGEUSDT", "ATOMUSDT", "NEOUSDT", "RUNEUSDT", "DASHUSDT", "VTHOUSDT", "DENTUSDT", "ONGUSDT", "OGNUSDT", "OXTUSDT", "BANDUSDT", "XVSUSDT", "SANDUSDT", "NMRUSDT", "CTSIUSDT", "CELRUSDT", "NKNUSDT", "SOLUSDT", "ETCUSDT", "AAVEUSDT", "ONEUSDT", "CRVUSDT", "IOSTUSDT", "CHRUSDT", "WINUSDT", "DUSKUSDT", "COTIUSDT", "FILUSDT", "BNTUSDT", "MANAUSDT", "SCUSDT", "VETUSDT", "TRXUSDT", "SUNUSDT"
]

interval = "1d"

start_date = "2019-10-01"
end_date = "2025-12-31"

start_timestamp = int(pd.Timestamp(start_date).timestamp() * 1000)
end_timestamp = int(pd.Timestamp(end_date).timestamp() * 1000)

all_pairs_data = []

for pair in pairs:

    print(f"Downloading {pair}...")

    start_time = start_timestamp

    all_data = []

    while start_time < end_timestamp:

        params = {
            "symbol": pair,
            "interval": interval,
            "startTime": start_time,
            "limit": 1000
        }

        data = requests.get(url, params=params).json()

        if len(data) == 0:
            break

        df = pd.DataFrame(data)

        all_data.append(df)

        start_time = data[-1][0] + 1

        time.sleep(0.2)

    pair_df = pd.concat(all_data, ignore_index=True)

    pair_df.columns = [
        "date",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "close_time",
        "quote_asset_volume",
        "num_trades",
        "taker_buy_base",
        "taker_buy_quote",
        "ignore"
    ]

    pair_df["date"] = pd.to_datetime(
        pair_df["date"],
        unit="ms"
    )

    pair_df = pair_df[["date", "close"]]

    pair_df = pair_df[
        pair_df["date"] < "2026-01-01"
    ]

    pair_df["Pair"] = pair

    all_pairs_data.append(pair_df)

crypto_prices = pd.concat(
    all_pairs_data,
    ignore_index=True
)

In [ ]:
crypto_prices = crypto_prices.sort_values(
["Pair","date"]
)

In [ ]:
crypto_prices["close"] = (
crypto_prices["close"].astype(float)
)

In [ ]:
# --- convert USDT prices to USD ---
fx = pd.read_excel("USDT_USD price per day.xlsx")
fx = fx.rename(columns={"Date": "date", "Price": "usdt_usd"})   
fx["date"] = pd.to_datetime(fx["date"]).dt.normalize()
fx = fx[["date", "usdt_usd"]].sort_values("date")

crypto_prices["date"] = pd.to_datetime(crypto_prices["date"]).dt.normalize()   
crypto_prices = crypto_prices.merge(fx, on="date", how="left")
crypto_prices["close"] = crypto_prices["close"] * crypto_prices["usdt_usd"]     
crypto_prices = crypto_prices.drop(columns="usdt_usd")                          

In [ ]:
  print("rows with no FX rate:", crypto_prices["close"].isna().sum())

In [ ]:
import numpy as np

crypto_prices["log_return"] = (
crypto_prices.groupby("Pair")["close"].transform(lambda x: np.log(x / x.shift(1)))
)

# Adding events and windows to the dataset - excluding event that were anticipated based on RQ4 and events with too little obs

In [ ]:
events = [
    {"name": "SEC_charges_ripple", "date": "2020-12-22"},
    {"name": "SEC_charges_binance", "date": "2023-06-05"},
    {"name": "CFTC_Charges_BitMEX", "date": "2020-10-01"},
    {"name": "SEC_charges_Kraken", "date": "2023-02-09"},
    {"name": "CFTC/SEC_joint_statement_spot_crypto_trading", "date": "2025-09-02"},
    {"name": "BTC_not_safe_word_tweet", "date": "2020-01-10"},
    {"name": "Buy_tesla_with_BTC_tweet", "date": "2021-03-24"},
    {"name": "Trump_tweet_america_crypto_capital", "date": "2024-09-30"},
    {"name": "ETH_transition_to_POS_tweet", "date": "2022-09-15"},
    {"name": "bybit_hack", "date": "2025-02-21"},
    {"name": "Ronin_bridge_hack", "date": "2022-03-29"},
    {"name": "polynetwork_hack", "date": "2021-08-10"},
    {"name": "Portal_attack", "date": "2022-02-02"},
    {"name": "DMM_Bitcoin_hack", "date": "2024-05-31"},
    {"name": "US_tarrifs_announcement", "date": "2025-04-02"},
    {"name": "FED_announces_QE", "date": "2020-03-23"},
    {"name": "Bank_of_japan_end_negative_rates", "date": "2024-03-19"},
]

crypto_prices["date"] = pd.to_datetime(crypto_prices["date"]).dt.normalize()

EST_START, EST_END = 90, 10   # estimation window: event-90 ... event-10
EVT_PRE, EVT_POST  = 0, 4     # event window:      event-2  ... event+4

events_sorted = sorted(events, key=lambda e: pd.Timestamp(e["date"]))

# --- Pass 1: event windows are protected and collected globally ---
event_windows = {}
all_event_window_dates = set()
for ev in events_sorted:
    d = pd.Timestamp(ev["date"])
    w = set(pd.date_range(d - pd.Timedelta(days=EVT_PRE),
                          d + pd.Timedelta(days=EVT_POST), freq="D"))
    event_windows[ev["name"]] = w
    clash = all_event_window_dates & w
    if clash:
        print(f"[warn] {ev['name']} event window overlaps another event window: {sorted(clash)}")
    all_event_window_dates |= w

# --- Pass 2: estimation windows, excluding any event window (of any event)
#     and any day already taken by an earlier event's estimation window ---
estimation_windows = {}
claimed_estimation_dates = set()
for ev in events_sorted:
    d = pd.Timestamp(ev["date"])
    full = set(pd.date_range(d - pd.Timedelta(days=EST_START),
                             d - pd.Timedelta(days=EST_END), freq="D"))
    clean = full - all_event_window_dates - claimed_estimation_dates
    estimation_windows[ev["name"]] = clean
    claimed_estimation_dates |= clean

# --- Write the columns (0 = outside, 1 = estimation, 2 = event) ---
for ev in events:
    name = ev["name"]
    crypto_prices[name] = 0
    crypto_prices.loc[crypto_prices["date"].isin(estimation_windows[name]), name] = 1
    crypto_prices.loc[crypto_prices["date"].isin(event_windows[name]),      name] = 2

# --- Summary ---
for ev in events_sorted:
    n = len(estimation_windows[ev["name"]])
    print(f"{ev['name']:<48} estimation days: {n:>2}  (event {ev['date']})")

# Excluding events with low number of observations - check if some were missed above

In [ ]:
MIN_EST_DAYS = 30

kept, dropped = [], []
for ev in events_sorted:
    n = len(estimation_windows[ev["name"]])
    (dropped if n < MIN_EST_DAYS else kept).append((ev["name"], ev["date"], n))

print(f"DROPPED (estimation days < {MIN_EST_DAYS}):")
for name, date, n in dropped:
    print(f"  {name:<48} {date}  days={n}")

print(f"\nKEPT ({len(kept)} events):")
for name, date, n in kept:
    print(f"  {name:<48} {date}  days={n}")

In [ ]:
MIN_EST_DAYS_LIMIT = 30   

events_to_remove = [name for name, days in estimation_windows.items()
                    if len(days) < MIN_EST_DAYS]

print("Dropping:", events_to_remove)
crypto_prices = crypto_prices.drop(columns=events_to_remove, errors="ignore")

# keep your events list in sync so later loops don't reference dropped events
events = [ev for ev in events if ev["name"] not in events_to_remove]

# Removing extreme observations

In [ ]:
to_drop = [
    ("2022-05-31", "LUNAUSDT"),
    ("2022-05-12", "LUNAUSDT"),
    ("2021-06-18", "SUNUSDT"),
    ("2022-05-11", "LUNAUSDT"),
    ("2022-05-13", "LUNAUSDT"),
]
drop_keys = {(pd.Timestamp(d).normalize(), p) for d, p in to_drop}

before = len(crypto_prices)
mask = [ (d, p) in drop_keys for d, p in zip(crypto_prices["date"], crypto_prices["Pair"]) ]
crypto_prices = crypto_prices[~pd.Series(mask, index=crypto_prices.index)].copy()

print(f"dropped {before - len(crypto_prices)} rows (expected 5)")

# Calculating abnormal returns

In [ ]:
events_df = pd.DataFrame(events)
event_columns = events_df["name"].tolist()

MIN_EST_OBS = 20          # skip a coin if it has too few estimation-window returns

all_results = []

for event in event_columns:

    # --- ESTIMATION WINDOW: per-coin mean (and sd) of log returns ---
    estimation_data = crypto_prices.loc[
        crypto_prices[event] == 1,
        ["date", "Pair", "log_return"]
    ].dropna()

    est_stats = (
        estimation_data
        .groupby("Pair")["log_return"]
        .agg(mean_est="mean", sd_est="std", n_est="count")
    )

    # keep only coins with enough estimation observations
    est_stats = est_stats[est_stats["n_est"] >= MIN_EST_OBS]

    # --- EVENT WINDOW: actual returns per coin/date ---
    event_data = crypto_prices.loc[
        crypto_prices[event] == 2,
        ["date", "Pair", "log_return"]
    ].dropna()

    # only coins that have BOTH a valid estimation mean and event-window data
    common_pairs = event_data["Pair"].unique()
    common_pairs = [p for p in common_pairs if p in est_stats.index]

    for pair in common_pairs:

        mean_est = est_stats.loc[pair, "mean_est"]
        sd_est   = est_stats.loc[pair, "sd_est"]
        n_est    = est_stats.loc[pair, "n_est"]

        temp = event_data.loc[event_data["Pair"] == pair, ["date", "log_return"]].copy()
        temp = temp.rename(columns={"log_return": "actual_return"})

        temp["expected_return"] = mean_est                          # mean-adjusted: E[R] = mean
        temp["abnormal_return"] = temp["actual_return"] - mean_est

        temp["event"]   = event
        temp["pair"]    = pair
        temp["mean_est"] = mean_est
        temp["sd_est"]   = sd_est      # sd of estimation returns -> for test statistics later
        temp["n_est"]    = n_est

        temp = temp[
            ["event", "date", "pair",
             "actual_return", "expected_return", "abnormal_return",
             "mean_est", "sd_est", "n_est"]
        ]

        all_results.append(temp)

# --- FINAL MASTER TABLE ---
master_ar_table = pd.concat(all_results, ignore_index=True)

print(master_ar_table.head())

# Adding event and coin mapping

In [ ]:
# --- Event-level group (unchanged: this one really is event -> group) ---
master_ar_table["group"] = master_ar_table["event"].map({
    "US_tarrifs_announcement": "Macro",
    "FED_announces_QE": "Macro",
    "Bank_of_japan_end_negative_rates": "Macro",
    "SEC_charges_ripple": "Regulatory",
    "SEC_charges_binance": "Regulatory",
    "CFTC_Charges_BitMEX": "Regulatory",
    "SEC_charges_Kraken": "Regulatory",
    "CFTC/SEC_joint_statement_spot_crypto_trading": "Regulatory",
    "bybit_hack": "Security & Exchange hacks",
    "Ronin_bridge_hack": "Security & Exchange hacks",
    "polynetwork_hack": "Security & Exchange hacks",
    "Portal_attack": "Security & Exchange hacks",
    "DMM_Bitcoin_hack": "Security & Exchange hacks",
    "BTC_not_safe_word_tweet": "Social postings",
    "Buy_tesla_with_BTC_tweet": "Social postings",
    "Trump_tweet_america_crypto_capital": "Social postings",
    "ETH_transition_to_POS_tweet": "Social postings",
})

# --- Coin group + expected direction: JOIN by event AND pair ---
coin_map = pd.read_excel("Mapping table for Coins.xlsx")
# align key column names to master_ar_table's ("event", "pair")
coin_map = coin_map.rename(columns={
    "Event": "event",
    "Pair": "pair",
    "Coin group": "coin_group",
    "expected direction": "expected_direction",
})
coin_map = coin_map[["event", "pair", "coin_group", "expected_direction"]]

master_ar_table = master_ar_table.merge(
    coin_map,
    on=["event", "pair"],
    how="left",          # keeps every AR row; unmapped coins get NaN coin_group
)

# --- sanity check: which AR rows didn't get a coin group? ---
unmatched = master_ar_table[master_ar_table["coin_group"].isna()]
print(f"AR rows without a coin group: {len(unmatched)}")
if len(unmatched):
    print(unmatched[["event", "pair"]].drop_duplicates().to_string(index=False))

# Calculating CAR and standardizing

In [ ]:
final_table = (
    master_ar_table
    .groupby(["event", "pair", "group", "coin_group", "expected_direction"])
    .agg(
        CAR=("abnormal_return", "sum"),        # cumulative abnormal return over the 5 days
        actual_return=("actual_return", "sum"),
        expected_return=("expected_return", "sum"),
        mean_est=("mean_est", "first"),        # constant per coin-event -> take one
        sd_est=("sd_est", "first"),
        n_est=("n_est", "first"),
        n_event_days=("date", "count"),        # how many days actually summed (should be 5)
    )
    .reset_index()
)

import numpy as np
final_table["SCAR"] = final_table["CAR"] / (final_table["sd_est"] * np.sqrt(final_table["n_event_days"]))

final_table["signed_SCAR"] = final_table["SCAR"] * final_table["expected_direction"]

print(final_table.head())

# Keeping coins with 5 day event window

In [ ]:
# keep only coins with the full 5-day event window (consistency for the √5 null and RQ1/RQ2)
before = len(final_table)
final_table = final_table[final_table["n_event_days"] == 5].copy()
print(f"kept {len(final_table)} event-coin pairs with full 2-day window; dropped {before - len(final_table)}")

# RESEARCH QUESTION 1

# PERMUTATION TEST - signed SCAR

In [ ]:
import numpy as np
from scipy.stats import f_oneway

np.random.seed(42)

# one value per event: average signed_SCAR across that event's coins
event_level = (final_table
               .groupby(["event", "group"])["signed_SCAR"]
               .mean()
               .reset_index(name="event_signed_SCAR"))

groups = event_level["group"].unique()
values = event_level["event_signed_SCAR"].values
labels = event_level["group"].values

# observed F across the event-type groups
obs_samples = [values[labels == g] for g in groups]
F_obs, _ = f_oneway(*obs_samples)

# permutation null: reshuffle group labels among the 22 events
n_perm = 10000
F_perm = np.empty(n_perm)
for i in range(n_perm):
    perm_labels = np.random.permutation(labels)
    perm_samples = [values[perm_labels == g] for g in groups]
    F_perm[i], _ = f_oneway(*perm_samples)

p_value = (np.sum(F_perm >= F_obs) + 1) / (n_perm + 1)

print("n events:", len(event_level))
print("Observed F:", round(F_obs, 4))
print("Permutation p-value:", round(p_value, 4))

In [ ]:
print(event_level)

In [ ]:
group_summary = (event_level
                 .groupby("group")["event_signed_SCAR"]
                 .agg(mean="mean", sd="std", n="count")
                 .reset_index())
print(group_summary)

# PERMUTATION TEST - absolute SCAR

In [ ]:
import numpy as np
from scipy.stats import f_oneway

np.random.seed(42)

# one value per event: average |SCAR| across that event's coins (magnitude, direction ignored)
event_level_abs = (final_table
                   .assign(abs_scar=final_table["SCAR"].abs())
                   .groupby(["event", "group"])["abs_scar"]
                   .mean()
                   .reset_index(name="event_abs_scar"))

groups_abs = event_level_abs["group"].unique()
values_abs = event_level_abs["event_abs_scar"].values
labels_abs = event_level_abs["group"].values

# observed F
obs_samples_abs = [values_abs[labels_abs == g] for g in groups_abs]
F_obs_abs, _ = f_oneway(*obs_samples_abs)

# permutation null
n_perm = 10000
F_perm_abs = np.empty(n_perm)
for i in range(n_perm):
    perm_labels = np.random.permutation(labels_abs)
    perm_samples = [values_abs[perm_labels == g] for g in groups_abs]
    F_perm_abs[i], _ = f_oneway(*perm_samples)

p_value_abs = (np.sum(F_perm_abs >= F_obs_abs) + 1) / (n_perm + 1)

print("ABSOLUTE SCAR (magnitude) test")
print("n events:", len(event_level_abs))
print("Observed F:", round(F_obs_abs, 4))
print("Permutation p-value:", round(p_value_abs, 4))

In [ ]:
group_summary = (event_level_abs
                 .groupby("group")["event_abs_scar"]
                 .agg(mean="mean", sd="std", n="count")
                 .reset_index())
print(group_summary)

# OLS with HC3 robust standard errors - signed SCAR

In [ ]:
import statsmodels.formula.api as smf

# event_level: columns event, group, event_signed_SCAR  (one row per event, ~22 rows)
# make the reference group explicit so coefficients read cleanly
model = smf.ols(
    'event_signed_SCAR ~ C(group, Treatment(reference="Macro"))',
    data=event_level
).fit(cov_type="HC3")

print(model.summary())

# tidy effect table: each group's difference from the reference, with 95% CIs
coefs = model.params
ci    = model.conf_int()
out = (coefs.to_frame("coef")
       .join(ci.rename(columns={0: "ci_low", 1: "ci_high"}))
       .join(model.pvalues.to_frame("p_value")))
print(out)

print("\nRobust overall F:", round(model.fvalue, 4), " p:", round(model.f_pvalue, 4))

# OLS with HC3 robust standard errors - absolute SCAR

In [ ]:
import statsmodels.formula.api as smf

# event_level_abs: one row per event, column event_abs_scar = mean |SCAR| across the event's coins
model_abs = smf.ols(
    'event_abs_scar ~ C(group, Treatment(reference="Macro"))',
    data=event_level_abs
).fit(cov_type="HC3")

print(model_abs.summary())

coefs = model_abs.params
ci    = model_abs.conf_int()
out_abs = (coefs.to_frame("coef")
           .join(ci.rename(columns={0: "ci_low", 1: "ci_high"}))
           .join(model_abs.pvalues.to_frame("p_value")))
print(out_abs)

print("\nRobust overall F:", round(model_abs.fvalue, 4), " p:", round(model_abs.f_pvalue, 4))